# NK recipes and interdependencies at N = 10, K = 1, 2, 4

Companion to `nk_qca.ipynb` (N = 5). At N = 10 full enumeration of matrices is out of reach (9^10 labeled matrices for K = 1), so each landscape gets its own random matrix (Levinthal's `Rand_row`) and results are reported by K. Polarity flips still need no reduction: contributions are i.i.d. and all statistics are flip invariant.

Not carried over from N = 5: the Walsh-support identifiability count and the matrix-class classifier, both need class enumeration.

In [1]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N = 10
K_LIST = [1, 2, 4]
M = 200            # landscapes (and random matrices) per K
rng = np.random.default_rng(1)

XINT = np.arange(2**N)
X = ((XINT[:, None] >> np.arange(N)[::-1]) & 1).astype(np.int8)     # column j = element j
NEIGH = XINT[:, None] ^ (1 << np.arange(N)[::-1])[None, :]           # one-flip neighbours, column j flips element j

def random_matrix(k, rng):
    """Unit diagonal, k random off-diagonal ones per row (Levinthal's Rand_row)."""
    A = np.eye(N, dtype=np.int8)
    for i in range(N):
        A[i, rng.choice([j for j in range(N) if j != i], k, replace=False)] = 1
    return A

def landscape(A, rng):
    f = np.zeros(2**N)
    for i in range(N):
        cols = np.flatnonzero(A[i])
        f += rng.random(2 ** len(cols))[X[:, cols] @ (2 ** np.arange(len(cols))[::-1])]
    return f / N

## 1. Prime implicants by dynamic programming

The 3^10 cubes sit in a (3,)*N array. A cube is an implicant if both halves along any free element are implicants. A cube is prime if freeing any fixed element breaks it. The assert checks that an additive f gives unate recipes.

In [2]:
# Cubes live in a (3,)*N array: index 0 or 1 = literal fixed to that value, 2 = element absent.
def implicants(ok):
    """ok: boolean over 2^N configurations. Returns, for every cube, whether all its configurations are ok."""
    T = ok.reshape((2,) * N)
    for a in range(N):
        T = np.concatenate([T, np.logical_and(T.take([0], a), T.take([1], a))], axis=a)
    return T

def prime_cubes(ok):
    T = implicants(ok)
    nonprime = np.zeros_like(T)
    for a in range(N):
        parent = T.take([2], a)                                         # the cube with element a freed
        sl = [slice(None)] * N; sl[a] = slice(0, 2)
        nonprime[tuple(sl)] |= np.broadcast_to(parent, T[tuple(sl)].shape)
    return np.argwhere(T & ~nonprime)                                   # rows: per-element code 0/1/2

def cube_cover(C):
    """Configurations covered by each cube (cubes x 2^N)."""
    fixed = C < 2
    return ((X[None, :, :] == C[:, None, :]) | ~fixed[:, None, :]).all(-1)

# sanity: with K = 0 style additive f, recipes are unate
f0 = X @ rng.random(N)
C0 = prime_cubes(f0 >= np.quantile(f0, 0.75))
assert not (((C0 == 0).any(0)) & ((C0 == 1).any(0))).any()

## 2. Features
Same definitions as `nk_qca.ipynb` sections 3, 8 and 9.

In [3]:
def interacting_pairs(A):
    S = (A.T.astype(int) @ A) > 0                                       # i, j share some row set
    np.fill_diagonal(S, False)
    return S

def recipe_features(f, A, Q):
    C = prime_cubes(f >= np.quantile(f, Q))
    fixed = C < 2
    nonunate = ((C == 0).any(0) & (C == 1).any(0)).sum()
    co = (fixed.T.astype(int) @ fixed) > 0; np.fill_diagonal(co, False)
    tru = interacting_pairs(A)
    iu = np.triu_indices(N, 1)
    co, tru = co[iu], tru[iu]
    return dict(n_primes=len(C), mean_len=fixed.sum(1).mean(), max_len=fixed.sum(1).max(),
                nonunate=nonunate, pair_base_rate=tru.mean(),
                false_conj=(co & ~tru).sum() / max(co.sum(), 1),
                recall=(co & tru).sum() / max(tru.sum(), 1))

def peak_features(f, Q):
    on = f >= np.quantile(f, Q)
    peaks = (f[:, None] > f[NEIGH]).all(1)
    C = prime_cubes(on); cov = cube_cover(C)
    per_prime = (cov & peaks).sum(1)
    left, cover = on.copy(), []
    while left.any():
        c = (cov & left).sum(1).argmax(); cover.append(c); left &= ~cov[c]
    per_cov = per_prime[cover]
    pk = np.flatnonzero(peaks & on)
    own = cov[per_prime == 1][:, pk].any(0)
    nlit = (C < 2).sum(1)
    return dict(peaks=peaks.sum(), peaks_on=len(pk), primes=len(C), cover=len(cover),
                primes_0pk=(per_prime == 0).mean(), cover_1pk=(per_cov == 1).mean(),
                peaks_isolated=own.mean() if len(pk) else np.nan,
                gmax_in_widest=cov[nlit == nlit.min(), f.argmax()].any())

def topq_partial(f, q, S, rng):
    obs = np.zeros(2**N, bool); obs[rng.choice(2**N, S, replace=False)] = True
    oi = np.flatnonzero(obs)
    on_obs = np.zeros(2**N, bool); on_obs[oi[np.argsort(f[oi])[-q:]]] = True
    C = prime_cubes(~(obs & ~on_obs))                                   # cubes that avoid observed failures
    cov = cube_cover(C); nlit = (C < 2).sum(1)
    left, pred = on_obs.copy(), np.zeros(2**N, bool)
    while left.any():
        c = ((cov & left).sum(1) * 100 - nlit).argmax(); pred |= cov[c]; left &= ~cov[c]
    top = np.zeros(2**N, bool); top[np.argsort(f)[-q:]] = True
    g = f.argmax()
    base = on_obs.copy(); base[rng.choice(np.flatnonzero(~obs), pred.sum() - on_obs.sum(), replace=False)] = True
    return dict(pred_size=pred.sum(), gmax_observed=obs[g], hit_gmax=pred[g], hit_gmax_base=base[g],
                recall_top=(pred & top).sum() / q, recall_top_base=(base & top).sum() / q,
                precision_top=(pred & top).sum() / pred.sum())

## 3. Simulate
Calibration Q = 0.75 for recipe features. Peaks at Q = 0.75, 0.9, 0.99. Partial tables observe S = 64 or 256 of 1024 configurations.

In [4]:
rows_r, rows_p, rows_t = [], [], []
for k in K_LIST:
    for m in range(M):
        A = random_matrix(k, rng); f = landscape(A, rng)
        rows_r.append(dict(K=k, **recipe_features(f, A, 0.75)))
        for Q in [0.75, 0.9, 0.99]:
            rows_p.append(dict(K=k, Q=Q, **peak_features(f, Q)))
        for S in [64, 256]:
            for q in [1, 3, 5]:
                rows_t.append(dict(K=k, S=S, q=q, **topq_partial(f, q, S, rng)))
rec = pd.DataFrame(rows_r); pk = pd.DataFrame(rows_p); tq = pd.DataFrame(rows_t)
rec.groupby("K").mean().round(3)

,n_primes,mean_len,max_len,nonunate,pair_base_rate,false_conj,recall
K,,,,,,,
1,68.650,6.213,8.000,6.005,0.212,0.788,0.998
2,107.060,6.827,8.675,9.465,0.515,0.485,1.000
4,174.085,7.692,9.310,10.000,0.929,0.071,1.000


## 4. Recover K from recipes

In [5]:
FEATS = ["n_primes", "mean_len", "max_len", "nonunate"]
def centroid_accuracy(df, label):
    idx = rng.permutation(len(df)); tr, te = df.iloc[idx[::2]], df.iloc[idx[1::2]]
    mu, sd = tr[FEATS].mean(), tr[FEATS].std().replace(0, 1)
    C = ((tr[FEATS] - mu) / sd).groupby(tr[label]).mean()
    Z = ((te[FEATS] - mu) / sd).to_numpy()
    pred = C.index[((Z[:, None, :] - C.to_numpy()[None]) ** 2).sum(-1).argmin(1)]
    return dict(accuracy=(pred == te[label].to_numpy()).mean(), chance=1 / df[label].nunique())
centroid_accuracy(rec, "K")

{'accuracy': np.float64(0.9233333333333333), 'chance': 0.3333333333333333}

## 5. Recipes and local maxima

In [6]:
pk.groupby(["Q", "K"]).mean().round(2)

peaks  peaks_on  primes  cover  primes_0pk  cover_1pk  peaks_isolated  \
Q    K                                                                          
0.75 1   3.40      3.39   68.65  46.56        0.34       0.54            0.99   
     2   7.68      7.62  107.06  54.48        0.43       0.48            1.00   
     4  22.44     22.03  174.08  76.04        0.44       0.49            0.98   
0.90 1   3.40      3.28   34.74  26.10        0.27       0.65            0.99   
     2   7.68      6.99   48.20  31.02        0.36       0.58            0.98   
     4  22.44     18.76   66.57  42.80        0.35       0.61            0.96   
0.99 1   3.40      2.23    5.46   5.08        0.09       0.90            1.00   
     2   7.68      3.32    6.53   5.83        0.15       0.82            0.98   
     4  22.44      5.90    7.82   7.30        0.10       0.89            0.98   

        gmax_in_widest  
Q    K                  
0.75 1            0.96  
     2            0.80  
     4            0.62  
0.90 1            0.95  
     2            0.85  
     4            0.71  
0.99 1            0.96  
     2            0.88  
     4            0.84

## 6. Global max and top q from partial tables

In [7]:
tq.groupby(["S", "q", "K"]).mean().round(2)

pred_size  gmax_observed  hit_gmax  hit_gmax_base  recall_top  \
S   q K                                                                  
64  1 1      56.24           0.06      0.35           0.14        0.35   
      2      61.28           0.06      0.32           0.13        0.32   
      4      58.40           0.06      0.29           0.12        0.29   
    3 1      91.92           0.06      0.66           0.14        0.63   
      2     103.15           0.09      0.57           0.15        0.52   
      4     115.38           0.04      0.34           0.16        0.34   
    5 1     132.43           0.06      0.82           0.17        0.76   
      2     142.20           0.06      0.64           0.20        0.61   
      4     161.34           0.06      0.46           0.20        0.42   
256 1 1      14.86           0.24      0.52           0.24        0.52   
      2      14.64           0.24      0.44           0.26        0.44   
      4      14.52           0.24      0.33           0.25        0.33   
    3 1      24.84           0.29      0.76           0.34        0.68   
      2      28.12           0.24      0.66           0.29        0.58   
      4      32.96           0.22      0.50           0.24        0.44   
    5 1      35.34           0.23      0.88           0.26        0.78   
      2      41.59           0.28      0.76           0.33        0.66   
      4      50.00           0.20      0.58           0.27        0.52   

         recall_top_base  precision_top  
S   q K                                  
64  1 1             0.14           0.01  
      2             0.13           0.01  
      4             0.12           0.01  
    3 1             0.13           0.03  
      2             0.15           0.02  
      4             0.14           0.01  
    5 1             0.17           0.03  
      2             0.19           0.02  
      4             0.20           0.01  
256 1 1             0.24           0.04  
      2             0.26           0.03  
      4             0.25           0.03  
    3 1             0.28           0.11  
      2             0.27           0.08  
      4             0.26           0.05  
    5 1             0.29           0.13  
      2             0.30           0.10  
      4             0.29           0.06